# Main Training Notebook

This notebook keeps reusable code in `src/et_severity` and uses the notebook only for setup, configuration, and experiment execution.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/12Gongsam/Multimodal-ET-Severity-Assessment.git"
REPO_NAME = "Multimodal-ET-Severity-Assessment"

def ensure_repo_checkout():
    cwd = Path.cwd().resolve()
    if cwd.name == "notebook" and (cwd.parent / "README.md").exists():
        repo_root = cwd.parent
    elif (cwd / "README.md").exists() and (cwd / "notebook").exists():
        repo_root = cwd
    elif (cwd / REPO_NAME / "README.md").exists():
        repo_root = cwd / REPO_NAME
    else:
        repo_root = cwd / REPO_NAME
        if not repo_root.exists():
            subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
        elif not (repo_root / "README.md").exists():
            raise FileNotFoundError(f"Repository directory exists but README.md is missing: {repo_root}")

    notebook_dir = repo_root / "notebook"
    if not notebook_dir.exists():
        raise FileNotFoundError(f"Notebook directory not found: {notebook_dir}")

    os.chdir(notebook_dir)
    return repo_root, notebook_dir

REPO_ROOT, NOTEBOOK_DIR = ensure_repo_checkout()
print(f"Repository root: {REPO_ROOT}")
print(f"Working directory: {NOTEBOOK_DIR}")


In [ ]:
%pip install -q -r ../requirements.txt

import sys
from pathlib import Path

SRC_DIR = (Path("..") / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Python executable: {sys.executable}")
print(f"Src path added: {SRC_DIR}")


In [ ]:
import pandas as pd

from et_severity import (
    DEFAULT_DEVICE,
    prepare_training_data,
    run_multimodal_severity_loso,
    run_single_modality_loso,
)


# Config


In [ ]:
DATA_ROOT = (NOTEBOOK_DIR / ".." / "data").resolve()
LABEL_CSV = DATA_ROOT / "relabel_md_k5.csv"
TARGET_COL = "target_k5"
FILTER_TASK = None
TARGET_PER_CLASS = 200
BATCH_SIZE = 16
DEVICE = DEFAULT_DEVICE

MULTIMODAL_CHECKPOINT_DIR = (NOTEBOOK_DIR / ".." / "checkpoint3").resolve()
SINGLE_CHECKPOINT_DIR = (NOTEBOOK_DIR / ".." / "checkpoint4").resolve()

print(f"Data root: {DATA_ROOT}")
print(f"Label csv: {LABEL_CSV}")
print(f"Device: {DEVICE}")


# Prepare Data


In [ ]:
manifest, loaders, split_result = prepare_training_data(
    root_dir=DATA_ROOT,
    label_csv_path=LABEL_CSV,
    target_col=TARGET_COL,
    target_per_class=TARGET_PER_CLASS,
    filter_task=FILTER_TASK,
    batch_size=BATCH_SIZE,
)

print(f"Manifest rows: {len(manifest)}")
print(f"Folds: {list(loaders.keys())}")
if split_result is not None:
    print(
        f"MultiDirect split summary: saved={len(split_result['saved'])}, "
        f"deleted={len(split_result['deleted'])}, kept={len(split_result['kept'])}, "
        f"errors={len(split_result['errors'])}"
    )

manifest.head()


# Multimodal Severity Example


In [ ]:
multimodal_results, multimodal_summary, multimodal_histories = run_multimodal_severity_loso(
    loaders,
    acc_name="LSTM",
    traj_name="ResNet18",
    device=DEVICE,
    epochs=200,
    learning_rate=1e-3,
    patience=20,
    checkpoint_dir=MULTIMODAL_CHECKPOINT_DIR,
)

multimodal_results


In [ ]:
multimodal_summary


# Single-Modality Severity Example


In [ ]:
single_results, single_summary, single_histories = run_single_modality_loso(
    loaders,
    model_name="MyWaveNet",
    modality="acc",
    target="severity",
    device=DEVICE,
    epochs=100,
    learning_rate=1e-3,
    patience=20,
    checkpoint_dir=SINGLE_CHECKPOINT_DIR,
)

single_results


In [ ]:
single_summary


# Optional Task Example


In [ ]:
# Uncomment this cell when you want to run task classification instead of severity.
#
# task_results, task_summary, task_histories = run_single_modality_loso(
#     loaders,
#     model_name="MyWaveNet",
#     modality="traj",
#     target="task",
#     device=DEVICE,
#     epochs=100,
#     learning_rate=1e-3,
#     patience=20,
#     checkpoint_dir=SINGLE_CHECKPOINT_DIR,
# )
#
# task_results
# task_summary
